# Validação de Precificação de Certificados de Recebíveis (`CRI` e `CRA`)

Este caderno documenta a validação da precificação fluxo a fluxo de Certificados de Recebíveis (`CertificadoRecebiveis`, `CRI` e `CRA`) contra os dados oficiais do portal ANBIMA Data (`CRICRAClient`). A amostra contempla ao menos dois ativos por indexador (`IPCA + Spread`, `DI + Spread`, `% do DI`, `IGP-M` e `PRÉ`).

In [3]:
import datetime, os
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
import pandas as pd
from bz_pricing.curves import CurveManager
from bz_pricing.data import CRICRAClient
from bz_pricing.instruments import CertificadoRecebiveis, CRI, CRA

client = CRICRAClient(data_dir='dados/cricra')
cm = CurveManager(allow_download=True)
tickers_anbima = [
    'CRA019003JY', 'CRA021000MG',   # IPCA + Spread
    '22D1289009',  'CRA024009Q2',   # DI + Spread
    'CRA024004H7', 'CRA0250018H', 'CRA026002BD',  # % do DI
]

## 1. Validação Contra Preços Oficiais do Mercado Secundário (ANBIMA Data)

Carregam-se os cadastros e os preços indicativos oficiais via `CRICRAClient` e confrontam-se PU, duration, DV01 e convexidade calculados pela biblioteca com os valores divulgados pela ANBIMA.

In [ ]:
rows = []
for tk in tickers_anbima:
    asset = CertificadoRecebiveis.from_ticker(tk, client=client)
    prices = client.get_asset_prices(tk)
    pu_ind = prices['pu_indicativo']
    pu_hist = prices.get('pu_historico') or {}
    ref_dt = datetime.date.fromisoformat(pu_ind['data_referencia'])
    taxa_ind = float(pu_ind['taxa_indicativa']) / 100.0
    pu_of = float(pu_ind['pu_indicativo'])
    dur_of = float(pu_ind['duration'])
    vna_of = float(pu_hist['vna']) if pu_hist.get('vna') else None
    pu_par_of = float(pu_hist['pu_par']) if pu_hist.get('pu_par') else None
    di_curve = cm.get_curve('PRE', ref_dt) if asset._is_di_linked() else None

    m = asset.risk_metrics(
        ref_dt,
        taxa_ind,
        vna=vna_of,
        di_curve=di_curve,
        pu_par_val=pu_par_of,
    )
    rows.append({
        'ticker': tk,
        'indexador': asset.indexador,
        'pct_di': asset._is_pct_di(),
        'ref_date': ref_dt,
        'pu_calc': m['pu'],
        'pu_anbima': pu_of,
        'diff_pu': m['pu'] - pu_of,
        'dur_calc': m['duration_du'],
        'dur_anbima': dur_of,
        'diff_dur': m['duration_du'] - dur_of,
        'dv01': m['dv01'],
        'convexity': m['convexity'],
    })

df_res = pd.DataFrame(rows)
df_res

,ticker,indexador,pct_di,ref_date,pu_calc,pu_anbima,diff_pu,dur_calc,dur_anbima,diff_dur,dv01,convexity
0,CRA019003JY,IPCA,False,2026-11-12,1411.241298,1411.241298,4.000003e-08,378.63,378.64,-0.01,0.196922,3.808194
1,CRA021000MG,IPCA,False,2026-11-12,1305.925395,1305.925395,2.000002e-08,327.92,327.92,0.00,0.156585,2.561756
2,22D1289009,DI,False,2026-11-12,1058.005593,1058.005594,-1.500000e-06,136.97,136.98,-0.01,0.050325,0.653607
3,CRA024009Q2,DI,False,2026-11-12,902.967139,902.954137,1.300136e-02,891.16,891.24,-0.08,0.268043,13.155948
4,CRA024004H7,DI,True,2026-11-12,1005.581251,1005.573782,7.469650e-03,526.85,526.88,-0.03,0.182723,5.149530
5,CRA0250018H,DI,True,2026-11-12,895.631423,895.656313,-2.488975e-02,688.95,689.01,-0.06,0.203985,7.782573
6,CRA026002BD,DI,True,2026-11-12,915.490190,915.470612,1.957788e-02,980.99,981.00,-0.01,0.296828,13.212821


## 2. Cobertura Complementar em `IGP-M` e Prefixado (`PRÉ`)

Validam-se dois ativos em `IGP-M` e dois em `PRÉ` construídos com `from_schedule(...)` nas convenções `BUS/252` e `ACT/360`, com verificação de consistência de PU, duration, DV01 e convexidade.

In [2]:
ref_sint = datetime.date(2026, 7, 1)
sinteticos = [
    CRI.from_schedule('CRI-IGPM-1', '2025-01-15', '2028-01-17', indexador='IGP-M', coupon_rate=0.065, coupon_freq='semestral'),
    CRA.from_schedule('CRA-IGPM-2', '2025-06-16', '2029-06-15', indexador='IGP-M', coupon_rate=0.070, coupon_freq='mensal', coupon_day_count='ACT/360'),
    CRI.from_schedule('CRI-PRE-1',  '2025-02-03', '2028-02-01', indexador='PRÉ',   coupon_rate=0.135, coupon_freq='semestral'),
    CRA.from_schedule('CRA-PRE-2',  '2025-03-03', '2027-09-01', indexador='PRÉ',   coupon_rate=0.142, coupon_freq='mensal'),
]
sint_rows = []
for a in sinteticos:
    vna_val = 1085.0 if 'IGP' in a.indexador else 1000.0
    taxa_m = 0.068 if 'IGP' in a.indexador else 0.138
    m = a.risk_metrics(ref_sint, taxa_m, vna=vna_val)
    sint_rows.append({'ticker': a.ticker, 'indexador': a.indexador, 'day_count': a.coupon_day_count, **m})
pd.DataFrame(sint_rows)

NameError: name 'datetime' is not defined

## 3. Referências Normativas e Bibliográficas

- **ANBIMA – Associação Brasileira das Entidades dos Mercados Financeiro e de Capitais.** *Metodologias ANBIMA de Precificação — Certificados de Recebíveis (CRA e CRI)*. Rio de Janeiro: ANBIMA, 2025. Disponível em: <https://www.anbima.com.br/pt_br/informar/precos-e-indices/precos/certificados-de-recebiveis.htm> e arquivo local `dados/manuais_anbima/Metodologias_ANBIMA_de_Precificacao_CRA_CRI_VF_c80ce9286c.pdf`.
- **ANBIMA – Associação Brasileira das Entidades dos Mercados Financeiro e de Capitais.** *Guia para Prestação de Informações na Precificação ANBIMA* e *Retificações de Certificados de Recebíveis*. Rio de Janeiro: ANBIMA. Arquivos locais em `dados/manuais_anbima/Guia_para_prestacao_de_informacoes_na_precificacao_ANBIMA_0b0eefd982.pdf` e `dados/manuais_anbima/retificacoes_certificados_de_recebiveis_1_e57fa189ba.xlsx`.
- **Banco Central do Brasil (BCB).** *Sistema Gerenciador de Séries Temporais (SGS) — Séries 12 (CDI), 433 (IPCA) e 189 (IGP-M)*. Brasília: BCB. Disponível em: <https://api.bcb.gov.br/>.
- **B3 S.A. – Brasil, Bolsa, Balcão.** *Taxas Referenciais (PRE e DIC)*. São Paulo: B3. Disponível em: <https://www.b3.com.br/>.